# Частина 2. Аналіз споживання електроенергії

Датасет **Individual household electric power consumption** містить хвилинні вимірювання споживання електроенергії одного домогосподарства у Франції протягом ~4 років (2006–2010).  

Основні стовпці:  
| Стовпець | Опис |
|---|---|
| `Global_active_power` | Загальна активна потужність (кВт) |
| `Global_reactive_power` | Реактивна потужність (кВт) |
| `Voltage` | Напруга (В) |
| `Global_intensity` | Сила струму (А) |
| `Sub_metering_1` | Кухня: бойлер, кондиціонер (Вт·год) |
| `Sub_metering_2` | Пральна/сушарка/холодильник (Вт·год) |
| `Sub_metering_3` | Водонагрівач та кондиціонер (Вт·год) |

In [ ]:
import pandas as pd
import timeit
import numpy as np

## 1. Завантаження датасету

Файл розділений крапкою з комою (`;`), дата і час об'єднуються в один стовпець `DateTime`.  
Пропущені значення позначені символом `?` - вони автоматично замінюються на `NaN`.

In [ ]:
df = pd.read_csv(
    "data/household_power_consumption.txt",
    sep=";",
    parse_dates={"DateTime": ["Date", "Time"]},
    dayfirst=True,
    na_values=["?"],
    low_memory=False
)
print(df.shape)
print(df.dtypes)
print(df.head())

## 2. Очищення даних

Пропущені значення (`NaN`) замінюються **медіаною** відповідного числового стовпця.  
Медіана обрана замість середнього, оскільки вона стійка до викидів (наприклад, аномально великих пікових значень потужності).

In [ ]:
def clean_power_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Очищає датасет: заповнює пропуски медіаною числових стовпців.
    """
    numeric_cols = df.select_dtypes(include='number').columns
    for col in numeric_cols:
        df[col].fillna(df[col].median(), inplace=True)
    return df

df = clean_power_data(df)
print(f"Пропуски після очищення: {df.isnull().sum().sum()}")

## 3. Фільтрація: потужність > 5 кВт

Відбираємо хвилини, коли сумарна активна потужність перевищувала **5 кВт** - пікове навантаження.  
Час виконання замірюється 10 разів за допомогою `timeit` для оцінки ефективності.

In [ ]:
def power_above_5kw(df: pd.DataFrame) -> pd.DataFrame:
    return df[df['Global_active_power'] > 5]

t = timeit.timeit(lambda: power_above_5kw(df), number=10)
res4 = power_above_5kw(df)
print(f"Записів з потужністю > 5 кВт: {len(res4)}")
print(f"Час виконання (10 ітерацій): {t:.4f} с")
print(res4.head())

## 4. Фільтрація за силою струму та порівняння груп

Вибираємо записи, де сила струму знаходиться в діапазоні **19–20 А** (близько до максимального навантаження на автомат 20 А).  
Далі залишаємо лише ті рядки, де споживання **Sub_metering_2** (пральна/холодильник) перевищує **Sub_metering_1** (кухня) - тобто «побутова» група споживала більше за «кухню».

In [ ]:
def current_filter(df: pd.DataFrame) -> pd.DataFrame:
    mask_current = df['Global_intensity'].between(19, 20)
    filtered = df[mask_current]
    # Sub_metering_1: кухня (бойлер, кондиціонер)
    # Sub_metering_2: пральна машина, сушарка, холодильник
    mask_appliance = filtered['Sub_metering_2'] > filtered['Sub_metering_1']
    return filtered[mask_appliance]

t = timeit.timeit(lambda: current_filter(df), number=10)
res5 = current_filter(df)
print(f"Результатів: {len(res5)}")
print(f"Час: {t:.4f} с")
print(res5.head())

## 5. Середні по вибірці 500 000 записів

Щоб уникнути зміщення через часові патерни (час доби, день тижня), беремо **випадкову** вибірку 500 000 записів (`random_state=42` для відтворюваності) та рахуємо середнє споживання по кожній із трьох груп лічильників.

Це дозволяє оцінити, яка група приладів загалом споживає більше.

In [ ]:
def random_sample_means(df: pd.DataFrame) -> pd.Series:
    sample = df.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

t = timeit.timeit(lambda: random_sample_means(df), number=5)
res6 = random_sample_means(df)
print("Середні по групах споживання (вибірка 500 000):")
print(res6)
print(f"Час (5 ітерацій): {t:.4f} с")

## 6. Вечірнє пікове споживання

Аналізуємо **вечірні** години (≥18:00) з потужністю > 6 кВт, де група **Sub_metering_2** є найбільшою серед трьох.  
Датасет розбивається на дві половини:
- Перша половина - беремо **кожен 3-й** запис.
- Друга половина - беремо **кожен 4-й** запис.

Таке розріджування знижує обсяг даних, зберігаючи рівномірний часовий розподіл.

In [ ]:
def evening_high_consumption(df: pd.DataFrame) -> pd.DataFrame:
    mask_time = df['DateTime'].dt.hour >= 18
    mask_power = df['Global_active_power'] > 6
    filtered = df[mask_time & mask_power]
    
    # Група 2 найбільша серед трьох
    mask_g2 = (
        (filtered['Sub_metering_2'] > filtered['Sub_metering_1']) &
        (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])
    )
    filtered = filtered[mask_g2].reset_index(drop=True)
    
    mid = len(filtered) // 2
    first_half  = filtered.iloc[:mid].iloc[::3]   # кожен 3-й
    second_half = filtered.iloc[mid:].iloc[::4]   # кожен 4-й
    
    return pd.concat([first_half, second_half])

t = timeit.timeit(lambda: evening_high_consumption(df), number=5)
res7 = evening_high_consumption(df)
print(f"Результатів: {len(res7)}")
print(f"Час: {t:.4f} с")
print(res7.head())

## 7. Нормалізація та стандартизація

Масштабування числових ознак необхідне для алгоритмів ML, чутливих до діапазону значень (наприклад, KNN, SVM, нейронні мережі).

| Метод | Формула | Діапазон | Коли використовувати |
|---|---|---|---|
| **Min-Max** (нормалізація) | `(x - min) / (max - min)` | [0, 1] | Коли потрібен фіксований діапазон |
| **Z-score** (стандартизація) | `(x - mean) / std` | [-∞, +∞] | Коли припускається нормальний розподіл |

In [ ]:
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage',
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

# Нормалізація (Min-Max)
df_normalized = df.copy()
for col in numeric_cols:
    mn, mx = df[col].min(), df[col].max()
    df_normalized[col] = (df[col] - mn) / (mx - mn)

# Стандартизація (Z-score)
df_standardized = df.copy()
for col in numeric_cols:
    df_standardized[col] = (df[col] - df[col].mean()) / df[col].std()

print("Нормалізований датасет (перші рядки):")
print(df_normalized[numeric_cols].head())
print("\nСтандартизований датасет (перші рядки):")
print(df_standardized[numeric_cols].head())

## 8. Кореляційний аналіз

Вимірюємо зв'язок між **загальною активною потужністю** і **напругою**:

- **Коефіцієнт Пірсона** - лінійна кореляція. Значення від -1 (повна обернена) до +1 (повна пряма). Чутливий до викидів.
- **Коефіцієнт Спірмена** - ранговa кореляція (монотонна залежність). Стійкий до викидів та нелінійних зв'язків.

Якщо обидва коефіцієнти схожі → зв'язок лінійний. Якщо Спірмен >> Пірсон → нелінійний монотонний зв'язок.

In [ ]:
col1, col2 = 'Global_active_power', 'Voltage'

pearson  = df[col1].corr(df[col2], method='pearson')
spearman = df[col1].corr(df[col2], method='spearman')

print(f"Коефіцієнт Пірсона  ({col1} vs {col2}): {pearson:.4f}")
print(f"Коефіцієнт Спірмена ({col1} vs {col2}): {spearman:.4f}")

## 9. One Hot Encoding (OHE) категоріального атрибуту

Категоріальний атрибут **«частина доби»** (`time_of_day`) не можна передавати в ML-моделі у вигляді рядка.  
Розбиваємо на чотири бінарні стовпці:
- `tod_morning` (6:00–12:00)
- `tod_afternoon` (12:00–18:00)
- `tod_evening` (18:00–22:00)
- `tod_night` (22:00–6:00)

В кожному рядку рівно одна з цих колонок = `True`, решта - `False`.

In [ ]:
# Додаємо категоріальний атрибут - частина доби
def time_of_day(hour):
    if 6 <= hour < 12:    return "morning"
    elif 12 <= hour < 18: return "afternoon"
    elif 18 <= hour < 22: return "evening"
    else:                 return "night"

df['time_of_day'] = df['DateTime'].dt.hour.apply(time_of_day)

df_ohe = pd.get_dummies(df, columns=['time_of_day'], prefix='tod')
print("Стовпці після One Hot Encoding:")
print([c for c in df_ohe.columns if c.startswith('tod')])
print(df_ohe.filter(like='tod').head())